In [2]:
# Transcription Accuracy
import pandas as pd
df = pd.read_csv("../data/sample_wer_comparison.csv")
df.head(50)

,chunk_id,third_party_transcript_cleaned,internal_transcript_cleaned,wer_third_party_cleaned_vs_internal_cleaned
0,CHUNK_001,Patient reports headache since yesterday and t...,Patient reports headache since yesterday and t...,0.1765
1,CHUNK_002,Follow-up appointment requested in two weeks a...,Follow-up appointment requested and bring results,0.2308
2,CHUNK_003,The patient reports no fever or cough during t...,The patient reports fever and cough during the...,0.3125
3,CHUNK_004,The physician recommends repeating the test if...,The physician recommends repeating the test if...,0.0000
4,CHUNK_005,Blood pressure has remained stable during trea...,Blood pressure remained stable during treatment,0.0286


In [3]:
#creating a new column for third party accuracy; error to quality
df["third_party_accuracy"] = (1 - df["wer_third_party_cleaned_vs_internal_cleaned"]) * 100
#show results
df[["wer_third_party_cleaned_vs_internal_cleaned", "third_party_accuracy"]].head()

,wer_third_party_cleaned_vs_internal_cleaned,third_party_accuracy
0,0.1765,82.35
1,0.2308,76.92
2,0.3125,68.75
3,0.0000,100.00
4,0.0286,97.14


In [4]:
len(df)

5

In [5]:
#quality AVG
df["third_party_accuracy"].mean().round(2)

np.float64(85.03)

In [6]:
df["third_party_accuracy"].describe().round(2)

count      5.00
mean      85.03
std       13.31
min       68.75
25%       76.92
50%       82.35
75%       97.14
max      100.00
Name: third_party_accuracy, dtype: float64

In [7]:
#cheking bug, WER should be 0-1
df["wer_third_party_cleaned_vs_internal_cleaned"].max()

np.float64(0.3125)

In [8]:
#filtering valid WER
df_clean = df[
    (df["wer_third_party_cleaned_vs_internal_cleaned"] >= 0) &
    (df["wer_third_party_cleaned_vs_internal_cleaned"] <= 1)
]
round(df_clean["wer_third_party_cleaned_vs_internal_cleaned"].max(),2)

np.float64(0.31)

In [9]:
# WER should be >= 0 & <= 1 only
df_clean = df[
    (df["wer_third_party_cleaned_vs_internal_cleaned"] >= 0) &
    (df["wer_third_party_cleaned_vs_internal_cleaned"] <= 1)
].copy()

In [10]:
df_clean.shape

(5, 5)

In [11]:
#Accuracy- third_party vs internal_column
df_clean["third_party_accuracy"] = (1 - df_clean["wer_third_party_cleaned_vs_internal_cleaned"]) * 100

df_clean["third_party_accuracy"].describe().round(2)

count      5.00
mean      85.03
std       13.31
min       68.75
25%       76.92
50%       82.35
75%       97.14
max      100.00
Name: third_party_accuracy, dtype: float64

In [12]:
df_clean.shape

(5, 5)

In [ ]:
# Merging Label Studio task ID
import json
with open("../data/sample_label_studio_export.json", "r", encoding="utf-8") as ls_json:
    data = json.load(ls_json)

ls_data = []

for task in data:
  task_id = task ["id"]
  chunk_id = task ["data"]["chunk_id"]
  ls_data.append({"chunk_id": chunk_id, "label_studio_id": task_id})

df_ls = pd.DataFrame(ls_data)

df_clean = df_clean.merge(df_ls, on="chunk_id", how="left")
df_clean.head(100)

FileNotFoundError: [Errno 2] No such file or directory: '/data/sample_label_studio_export.json'

In [ ]:
#Mapping with LS
df_clean.shape
df_clean["label_studio_id"].isna().mean()

In [ ]:
# csv for audit
df_audit = df_clean.sort_values(by="third_party_accuracy", ascending=True)[[
    "label_studio_id",
    "third_party_transcript_cleaned",
    "internal_transcript_cleaned",
    "third_party_accuracy",
    "wer_third_party_cleaned_vs_internal_cleaned",
]]
df_audit.to_csv("../reports/sample_qa_audit.csv", index=False)